# SAM3 Basketball Segmentation
Run `run_sam3.py` on Google Colab with GPU acceleration.

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Paths — fill these in ─────────────────────────────────────────────────
PROJECT_DIR   = "/content/drive/MyDrive/sam3-experiments"  # folder containing run_sam3.py and sam3.pt
VIDEO_PATH    = "/content/drive/MyDrive/sam3-experiments/data/videos/djurgarden1.mp4"         # input video
OUTPUT_DIR    = "/content/drive/MyDrive/sam3-experiments/output"            # where the annotated video will be saved

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install "ultralytics==8.4.21" opencv-python-headless -q

In [ ]:
# ── 4. Verify GPU ────────────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── 5. Copy video to local storage for faster I/O ────────────────────────────
import os, shutil
from pathlib import Path

video_local = Path("/content") / Path(VIDEO_PATH).name
if not video_local.exists():
    print(f"Copying {VIDEO_PATH} → {video_local} ...")
    shutil.copy(VIDEO_PATH, video_local)
    print("Done.")
else:
    print(f"Already exists: {video_local}")

In [ ]:
# ── 6. Run SAM3 ──────────────────────────────────────────────────────────────
import subprocess, sys

script  = str(Path(PROJECT_DIR) / "run_sam3.py")
model   = str(Path(PROJECT_DIR) / "sam3.pt")
out_dir = OUTPUT_DIR

os.makedirs(out_dir, exist_ok=True)
os.environ["SAM3_OUTPUT_DIR"] = out_dir

cmd = [
    sys.executable, script,
    str(video_local),
    "--model", model,
    "--half",
]
print("Running:", " ".join(cmd), "\n")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)

process.wait()
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, cmd)